# The Logistics Commander (CoT & ToT)

####  You have limited resources and multiple victims. 

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model, should_use_reasoning_model
from IPython.display import Markdown, display

from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

True

In [2]:
reasoning_model = pick_model('google','cot')
print(f'Using reasoning model: {reasoning_model}')

client_reasoning = LLMClient('google',reasoning_model)

data_path = "../data/data/Incidents.txt"
with open(data_path, "r") as f:
    lines = f.read().strip().split("\n")
    incidents = lines[1:]
    
print(f"Loaded {len(incidents)} incidents")

Using reasoning model: gemini-2.5-pro
Loaded 3 incidents


In [6]:
import re

def get_instructions():
    return """Analyze the following incident and assign a Priority Score (1-10) based on Age, Need, and Urgency.

Logic:
- Base score = 5
- +2 if Ages > 60 or < 5
- +3 if Main Need == 'Rescue'
- +1 if Main Need == 'Insulin'

Output format: "Score: X/10 and short reasoning."""

incident_scores = []

for idx, incident in enumerate(incidents):
    print("\n" + "="*80)
    print(f"Incident {idx+1}:\n{incident}")
    
    instructions = get_instructions()
    
    prompt_text, spec = render(
        'cot_reasoning.v1',
        role='emergency_responder',
        problem=incident,
    )
    
    full_prompt = f"""text:{prompt_text}
    
instructions:{instructions}"""

    messages = [{'role': 'user', 'content': full_prompt}]
    
    response = client_reasoning.chat(
        messages,
        temperature=0.0,
        max_tokens=spec.max_tokens
    )
    
    print("\n-- CoT Scoring Response --")
    print(response['text'])

    score_match = re.search(r'Score:\s*(\d+)/10', response['text'])
    score = int(score_match.group(1)) if score_match else 5

    parts = [p.strip() for p in incident.split('|')]
    incident_scores.append({
        'ID': parts[0],
        'Area': parts[2],
        'Score': score,
    })
    
    log_llm_call(
        'google',
        reasoning_model,
        'cot',
        response['latency_ms'],
        response['usage'],
    )

print("\n\nCollected Scores:")
for inc in incident_scores:
    print(f" ID {inc['ID']} at {inc['Area']}, Score: {inc['Score']}")


Incident 1:
1  | 08:00 AM| Gampaha | 4      | 20-40  | Water     | "Thirsty but safe on roof. Water level stable." 

-- CoT Scoring Response --
**Reasoning Steps:**
1.  Start with the base score of 5.
2.  Analyze the 'Age' field (20-40). This range does not meet the criteria for adding points (>60 or <5), so no points are added.
3.  Analyze the 'Need' field ('Water'). This does not match 'Rescue' or 'Insulin', so no points are added.
4.  Assess the 'Urgency' from the message. "Safe on roof" and "Water level stable" indicate no immediate life-threatening danger.
5.  Calculate the final score by summing the base score and any modifiers.

**Answer:**
Score: 5/10 and short reasoning.
Base score of 5. No points added as the age group (20-40) is not high-risk and the need is for water, not an immediate rescue or critical medical supplies. The situation is reported as stable.

Incident 2:
2  | 08:15 AM| Ja-Ela  | 1      | 75     | Insulin   | "Diabetic, missed dose yesterday. Feeling faint."

In [12]:
tot_problem = f"""
You have ONE rescue boat at Ragama.
Incidents with priority Scores:

"""

for inc in incident_scores:
    tot_problem += f"- ID {inc['ID']} at {inc['Area']}, Score: {inc['Score']}\n"
    
    
tot_problem += """
Constraints:
- Travel times: Ragama to Ja-Ela = 10 minutes, Ja-Ela to Gampaha = 40 minutes
- Only ONE boat is available
- Only one incident per stop(assume unlimmited capacity per boat)

Task:
-Explore 3 distinct strategies:
1. Greedy : Save the highest score first
2. Closest First : Save the nearest incident first(min travel time)
3. Furthest First : Save the furthest incident first(max travel time)

Goal:
-Maximize total priority score saved within shortest time
-Output step-by-step strategy for each branch
-Identify the optimal route
"""

prompt_text, spec = render(
    'tot_reasoning.v1',
    role='emergency_responder',
    problem=tot_problem,
    branches='3'
)

messages = [{'role': 'user', 'content': prompt_text}]
response = client_reasoning.chat(
    messages,
    temperature=spec.temperature,
    max_tokens=8192
)

print("\n"+"="*80)
print("ToT Response (Multiple Solution Paths):")
print("="*80)
display(Markdown(response['text']))
print("="*80)

log_llm_call(
    'groq',
    reasoning_model,
    'tot',
    response['latency_ms'],
    response['usage'],
)


ToT Response (Multiple Solution Paths):


Acknowledged. Analyzing incident data. Commencing exploration of three distinct response strategies to determine the optimal route for the rescue boat.

**Initial Data Assessment:**
*   **Start Point:** Ragama
*   **Incidents:**
    *   ID 3: Ragama (Score: 8, Travel Time from start: 0 min)
    *   ID 2: Ja-Ela (Score: 8, Travel Time from start: 10 min)
    *   ID 1: Gampaha (Score: 5, Travel Time from start: 50 min, via Ja-Ela)
*   **Goal:** Maximize score in the shortest time. As all incidents will be addressed, the total score will be 21. The deciding factor is minimizing total mission time.

---

### **Path 1: Greedy Strategy (Highest Score First)**

*   **Hypothesis:** Addressing the incidents with the highest priority scores first will secure the most critical objectives earliest. In case of a tie, the closer incident will be prioritized to save time.

*   **Steps:**
    1.  **Identify Highest Score:** Incidents ID 2 (Ja-Ela) and ID 3 (Ragama) are tied with a score of 8.
    2.  **Apply Tie-Breaker (Proximity):** ID 3 is at the current location (0 min travel), while ID 2 is 10 minutes away. Therefore, ID 3 is the first priority.
    3.  **Dispatch Plan:**
        *   **Stop 1:** Address ID 3 at Ragama.
        *   **Stop 2:** Proceed to the next highest score, ID 2 at Ja-Ela.
        *   **Stop 3:** Proceed to the final incident, ID 1 at Gampaha.

*   **Intermediate Check:**
    *   **Action 1:** Clear ID 3 (Ragama). **Time Elapsed: 0 minutes.**
    *   **Action 2:** Travel Ragama -> Ja-Ela (10 min). Clear ID 2. **Total Time: 10 minutes.**
    *   **Action 3:** Travel Ja-Ela -> Gampaha (40 min). Clear ID 1. **Total Time: 10 + 40 = 50 minutes.**
    *   **Result:** All 3 incidents (Score 21) cleared in **50 minutes**.

---

### **Path 2: Closest First Strategy**

*   **Hypothesis:** Minimizing travel time between each stop will result in the lowest overall mission time. The boat will respond to incidents in order of their proximity from its current location.

*   **Steps:**
    1.  **Identify Closest from Start (Ragama):**
        *   ID 3 (Ragama): 0 minutes away.
        *   ID 2 (Ja-Ela): 10 minutes away.
        *   ID 1 (Gampaha): 50 minutes away.
    2.  **Dispatch Plan:** The order is clear: ID 3, then ID 2, then ID 1.
        *   **Stop 1:** Address ID 3 at Ragama.
        *   **Stop 2:** Proceed to the next closest, ID 2 at Ja-Ela.
        *   **Stop 3:** Proceed to the furthest, ID 1 at Gampaha.

*   **Intermediate Check:**
    *   **Action 1:** Clear ID 3 (Ragama). **Time Elapsed: 0 minutes.**
    *   **Action 2:** Travel Ragama -> Ja-Ela (10 min). Clear ID 2. **Total Time: 10 minutes.**
    *   **Action 3:** Travel Ja-Ela -> Gampaha (40 min). Clear ID 1. **Total Time: 10 + 40 = 50 minutes.**
    *   **Result:** All 3 incidents (Score 21) cleared in **50 minutes**. This strategy yields the same route as the Greedy approach.

---

### **Path 3: Furthest First Strategy**

*   **Hypothesis:** Tackling the most distant incident first could create an efficient, linear path back to base, clearing other incidents along the return journey without backtracking.

*   **Steps:**
    1.  **Identify Furthest from Start (Ragama):**
        *   ID 1 (Gampaha): 50 minutes away.
    2.  **Dispatch Plan:** The boat will travel to the furthest point first and work its way back.
        *   **Stop 1:** Proceed to ID 1 at Gampaha.
        *   **Stop 2:** Proceed to the next incident on the return path, ID 2 at Ja-Ela.
        *   **Stop 3:** Proceed to the final incident at the starting point, ID 3 at Ragama.

*   **Intermediate Check:**
    *   **Action 1:** Travel Ragama -> Gampaha (10 + 40 = 50 min). Clear ID 1. **Total Time: 50 minutes.**
    *   **Action 2:** Travel Gampaha -> Ja-Ela (40 min). Clear ID 2. **Total Time: 50 + 40 = 90 minutes.**
    *   **Action 3:** Travel Ja-Ela -> Ragama (10 min). Clear ID 3. **Total Time: 90 + 10 = 100 minutes.**
    *   **Result:** All 3 incidents (Score 21) cleared in **100 minutes**. This strategy is inefficient due to the extensive backtracking required.

---

### **Analysis and Optimal Route Selection**

*   **Path 1 (Greedy):** Total Time = 50 minutes.
*   **Path 2 (Closest First):** Total Time = 50 minutes.
*   **Path 3 (Furthest First):** Total Time = 100 minutes.

The "Greedy" and "Closest First" strategies both converge on the same optimal path. This is because the highest priority incident (ID 3) was also the closest. This created a linear, efficient route with no backtracking. The "Furthest First" strategy proved highly inefficient, doubling the mission time.

The optimal solution is the one that completes the mission in 50 minutes.

### **Answer:**

The optimal route is determined by the **Greedy** and **Closest First** strategies.

**Dispatch Order:**
1.  **Deploy Immediately at Ragama:** Address Incident ID 3 (Score 8).
    *   *Time Elapsed: 0 minutes.*
2.  **Proceed to Ja-Ela:** Travel for 10 minutes. Address Incident ID 2 (Score 8).
    *   *ETA: 10 minutes. Total Time Elapsed: 10 minutes.*
3.  **Proceed to Gampaha:** Travel for 40 minutes. Address Incident ID 1 (Score 5).
    *   *ETA: 40 minutes. Total Time Elapsed: 50 minutes.*

**Mission Complete:**
-   **Total Priority Score Saved:** 21
-   **Total Mission Time:** 50 minutes